### Merging the files

In [1]:
import pandas as pd
import json

In [2]:
# def parse_triplet_text(text):
#     """
#     Parse triplet text that is wrapped in <python> tags and contains a JSON-like list structure.
    
#     Args:
#         text (str): The text containing triplets in format:
#                    "<python>\n[[subject, predicate, object], ...]\n</python>"
    
#     Returns:
#         list: A list of triplets, where each triplet is a list of [subject, predicate, object]
    
#     Example:
#         >>> text = '<python>\\n[["A", "is", "B"]]\\n</python>'
#         >>> parse_triplet_text(text)
#         [['A', 'is', 'B']]
#     """
#     import json
    
#     # Remove the <python> and </python> tags
#     text = text.strip()
#     if text.startswith("<python>"):
#         text = text[8:]  # Remove "<python>"
#     if text.endswith("</python>"):
#         text = text[:-9]  # Remove "</python>"
    
#     # Strip whitespace
#     text = text.strip()
    
#     # Parse the JSON list
#     try:
#         triplets = json.loads(text)
#         return triplets
#     except json.JSONDecodeError as e:
#         print(f"Error parsing JSON: {e}")
#         return []


def parse_triplet_text(text):
    """
    Parse triplet text that may be embedded anywhere in a larger text and wrapped
    in <python>...</python> tags.

    Args:
        text (str): Text containing a <python> block with a JSON-like list of triplets.

    Returns:
        list: A list of triplets [subject, predicate, object]
    """
    import json
    import re

    if not isinstance(text, str):
        return []

    # Regex to extract content inside <python>...</python>
    match = re.search(r"<python>\s*(.*?)\s*</python>", text, re.DOTALL) or re.search(r"```python\s*(.*?)\s*```", text, re.DOTALL)

    if not match:
        # No python block found
        return []

    python_block = match.group(1).strip()

    try:
        triplets = json.loads(python_block)
        return triplets
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []



In [6]:
data = pd.read_csv("../data/hallucination_test/summcoz_train_sample_100.csv")
data.tail()

,id,summary,article,reason,origin,dataset,label
95,95,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states that Stuart Pearce belie...,cnndm,cogensumm,2
96,96,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states neville has been in char...,cnndm,cogensumm,2
97,97,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary claims neville has been tipped ...,cnndm,cogensumm,2
98,98,stuart pearce says neville needs to get manage...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states neville has been tipped ...,cnndm,cogensumm,2
99,99,the double world champion has chosen shrewdly ...,Being in the right place at the right time is ...,1. The summary claims hamilton says his new me...,cnndm,cogensumm,2


In [7]:
# CORRECT FIX - Track offset for each file
# Text and summary data
# data_documents = pd.read_csv("../data/data_documents.csv")
# data_summaries = pd.read_csv("../data/data_summaries.csv")

# # Merge the dataframes
# filterd_df = data_documents.merge(data_summaries, on='id', how='inner')[:300]

filterd_df = data.copy()
# Load batch results - properly handle offset for each file
batch_results_ver = [1]  # List of versions to process
batch_results_dict = {}

offset = 0  # Track cumulative offset across files

for ver in batch_results_ver:
    documents = "documents"
    
    with open(f"../data/hallucination_test/batch_results_{documents}_{ver}.jsonl", 'r') as f:
        print(f"Processing batch_results_{documents}_{ver}.jsonl (offset: {offset})")
        
        for idx, line in enumerate(f):
            result = dict(json.loads(line))
            item = result['result']['message']['content']
            text_content = item[0]['text']
            triplets = parse_triplet_text(text_content)
            
            # CRITICAL FIX: Add offset to get the correct row from merged_df
            row_index = offset + idx
            
            # Get the actual text_id from the correct row
            if row_index < len(filterd_df):
                text_id = filterd_df.iloc[row_index]['id']
                
                # Initialize list if first time seeing this ID
                if text_id not in batch_results_dict:
                    batch_results_dict[text_id] = []
                
                # Add triplets
                batch_results_dict[text_id].extend(triplets)
    
    # Update offset for next file
    # Count how many lines were in this file
    with open(f"../data/hallucination_test/batch_results_{documents}_{ver}.jsonl", 'r') as f:
        lines_in_file = sum(1 for _ in f)
    offset += lines_in_file

print(f"\n✓ Aggregated triplets for {len(batch_results_dict)} unique IDs.")

# Convert aggregated results to DataFrame
batch_results = [{'id': text_id, 'document_triplets': triplets} 
                         for text_id, triplets in batch_results_dict.items()]
batch_results_df = pd.DataFrame(batch_results)

# Merge with the original data
merged_df = filterd_df.merge(batch_results_df, on='id', how='left')

# Verify results
print(f"\n✅ CORRECTED VERSION:")
print(f"  Total rows: {len(merged_df)}")
print(f"  Unique IDs: {merged_df  ['id'].nunique()}")
print(f"  Rows with triplets: {merged_df['document_triplets'].notna().sum()}")
print(f"  Rows without triplets: {merged_df['document_triplets'].isna().sum()}")

display(merged_df.tail(10))

Processing batch_results_documents_1.jsonl (offset: 0)

✓ Aggregated triplets for 100 unique IDs.

✅ CORRECTED VERSION:
  Total rows: 100
  Unique IDs: 100
  Rows with triplets: 100
  Rows without triplets: 0


,id,summary,article,reason,origin,dataset,label,document_triplets
90,90,"jacqueline crane, 52, pretended to be a profes...",Jacqueline Crane (pictured outside court) clai...,"1. The summary states jacqueline crane, 52, pr...",cnndm,cogensumm,2,"[[Jacqueline Crane, claimed to be, professor o..."
91,91,"jacqueline crane, 52, pretended to be a profes...",Jacqueline Crane (pictured outside court) clai...,"1. The summary says jacqueline crane, 52, pret...",cnndm,cogensumm,2,"[[Jacqueline Crane, claimed to be, professor o..."
92,92,taylor swift has been a star since the age of ...,Taylor Swift has been a star since the age of ...,1. The summary states that Taylor Swift has 52...,cnndm,cogensumm,2,"[[Taylor Swift, has been, star], [Taylor Swift..."
93,93,taylor swift has been a fan since the age of 1...,Taylor Swift has been a star since the age of ...,1. The summary states taylor swift has been a ...,cnndm,cogensumm,2,"[[Taylor Swift, has been, star], [Taylor Swift..."
94,94,taylor swift has been a star since the age of ...,Taylor Swift has been a star since the age of ...,1. The summary claims the 25 year old from pen...,cnndm,cogensumm,2,"[[Taylor Swift, has been, star], [Taylor Swift..."
95,95,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states that Stuart Pearce belie...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ..."
96,96,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states neville has been in char...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ..."
97,97,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary claims neville has been tipped ...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ..."
98,98,stuart pearce says neville needs to get manage...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states neville has been tipped ...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ..."
99,99,the double world champion has chosen shrewdly ...,Being in the right place at the right time is ...,1. The summary claims hamilton says his new me...,cnndm,cogensumm,2,"[[Lewis Hamilton, is, Formula One driver], [Le..."


In [9]:
# # CORRECT FIX - Track offset for each file
# # Text and summary data
# data_documents = pd.read_csv("../data/data_documents.csv")
# data_summaries = pd.read_csv("../data/data_summaries.csv")

# # Merge the dataframes
# filterd_df = data_documents.merge(data_summaries, on='id', how='inner')[:300]   
# Load batch results - properly handle offset for each file
batch_results_ver = [1]  # List of versions to process
batch_results_dict = {}

offset = 0  # Track cumulative offset across files

for ver in batch_results_ver:
    documents = "summaries"
    
    with open(f"../data/hallucination_test/batch_results_{documents}_{ver}.jsonl", 'r') as f:
        print(f"Processing batch_results_{documents}_{ver}.jsonl (offset: {offset})")
        
        for idx, line in enumerate(f):
            result = dict(json.loads(line))
            item = result['result']['message']['content']
            text_content = item[0]['text']
            triplets = parse_triplet_text(text_content)
            
            # CRITICAL FIX: Add offset to get the correct row from merged_df
            row_index = offset + idx
            
            # Get the actual text_id from the correct row
            if row_index < len(filterd_df):
                text_id = filterd_df.iloc[row_index]['id']
                
                # Initialize list if first time seeing this ID
                if text_id not in batch_results_dict:
                    batch_results_dict[text_id] = []
                
                # Add triplets
                batch_results_dict[text_id].extend(triplets)
    
    # Update offset for next file
    # Count how many lines were in this file
    with open(f"../data/hallucination_test/batch_results_{documents}_{ver}.jsonl", 'r') as f:
        lines_in_file = sum(1 for _ in f)
    offset += lines_in_file

print(f"\n✓ Aggregated triplets for {len(batch_results_dict)} unique IDs.")

# Convert aggregated results to DataFrame
batch_results = [{'id': text_id, 'gen_summary_triplets': triplets} 
                         for text_id, triplets in batch_results_dict.items()]
batch_results_df = pd.DataFrame(batch_results)

# Merge with the original data
triplets_df = merged_df.merge(batch_results_df, on='id', how='left')

# Verify results
print(f"\n✅ CORRECTED VERSION:")
print(f"  Total rows: {len(triplets_df)}")
print(f"  Unique IDs: {triplets_df['id'].nunique()}")
print(f"  Rows with triplets: {triplets_df['gen_summary_triplets'].notna().sum()}")
print(f"  Rows without triplets: {triplets_df['gen_summary_triplets'].isna().sum()}")

display(triplets_df.tail(10))

Processing batch_results_summaries_1.jsonl (offset: 0)

✓ Aggregated triplets for 100 unique IDs.

✅ CORRECTED VERSION:
  Total rows: 100
  Unique IDs: 100
  Rows with triplets: 100
  Rows without triplets: 0


,id,summary,article,reason,origin,dataset,label,document_triplets,gen_summary_triplets
90,90,"jacqueline crane, 52, pretended to be a profes...",Jacqueline Crane (pictured outside court) clai...,"1. The summary states jacqueline crane, 52, pr...",cnndm,cogensumm,2,"[[Jacqueline Crane, claimed to be, professor o...","[[Jacqueline Crane, age, 52], [Jacqueline Cran..."
91,91,"jacqueline crane, 52, pretended to be a profes...",Jacqueline Crane (pictured outside court) clai...,"1. The summary says jacqueline crane, 52, pret...",cnndm,cogensumm,2,"[[Jacqueline Crane, claimed to be, professor o...","[[Jacqueline Crane, is, 52 years old], [Jacque..."
92,92,taylor swift has been a star since the age of ...,Taylor Swift has been a star since the age of ...,1. The summary states that Taylor Swift has 52...,cnndm,cogensumm,2,"[[Taylor Swift, has been, star], [Taylor Swift...","[[Taylor Swift, has been, star], [Taylor Swift..."
93,93,taylor swift has been a fan since the age of 1...,Taylor Swift has been a star since the age of ...,1. The summary states taylor swift has been a ...,cnndm,cogensumm,2,"[[Taylor Swift, has been, star], [Taylor Swift...","[[Taylor Swift, has been, fan since age 16], [..."
94,94,taylor swift has been a star since the age of ...,Taylor Swift has been a star since the age of ...,1. The summary claims the 25 year old from pen...,cnndm,cogensumm,2,"[[Taylor Swift, has been, star], [Taylor Swift...","[[Taylor Swift, has been, star], [Taylor Swift..."
95,95,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states that Stuart Pearce belie...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ...","[[Stuart Pearce, believes, Gary Neville], [Gar..."
96,96,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states neville has been in char...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ...","[[Stuart Pearce, believes, Gary Neville], [Gar..."
97,97,stuart pearce believes gary neville needs to g...,Stuart Pearce believes Gary Neville needs to g...,1. The summary claims neville has been tipped ...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ...","[[Stuart Pearce, believes, Gary Neville], [Gar..."
98,98,stuart pearce says neville needs to get manage...,Stuart Pearce believes Gary Neville needs to g...,1. The summary states neville has been tipped ...,cnndm,cogensumm,2,"[[Stuart Pearce, believes, Gary Neville needs ...","[[Stuart Pearce, says, Neville needs to get ma..."
99,99,the double world champion has chosen shrewdly ...,Being in the right place at the right time is ...,1. The summary claims hamilton says his new me...,cnndm,cogensumm,2,"[[Lewis Hamilton, is, Formula One driver], [Le...","[[double world champion, is, Sebastian Vettel]..."


In [10]:
len(batch_results_df)

100

In [12]:
# save the dataframe
triplets_df.to_csv("../data/hallucination_test/data_with_triplets.csv", index=False)